# ModernTCN — OFAT sensitivity at h = 5, custom anchor

One-factor-at-a-time sensitivity for **ModernTCN at h = 5 only**, centred on the
anchor you supplied (not the one in `tuningresults/`). All **13** hyperparameters
in the search space are pinned at that anchor and one is varied at a time; each
point trains `ITR` seeds, so every curve carries a mean ± std.

### Anchor

```
seq_len 35 · patch_size 32 · patch_stride 2 · ffn_ratio 2 · num_blocks 1
large_size 31 · small_size 5 · dim 64 · dropout 0 · head_dropout 0
revin 1 · learning_rate 0.0077943332090161695 · batch_size 256
```

Target and split are the current ones — `Y_t^(h) = ln((1/5) Σ RV_{t+k})` on
`data/EURUSD-RV.csv`, 643 test rows.

Results land in their **own** files (`ofat_moderntcn_h5_custom.csv`,
`sensitivity/figures/h5_custom/`), so nothing collides with the tuned-anchor study.
The anchor is saved beside the CSV, so a resumed sweep can never mix points from
two different centres.

> **41 configs × `ITR` seeds.** Resumable — completed `(param, value)` points are
> skipped, so you can stop, reconnect and re-run the sweep cell. Set
> `USE_DRIVE = True` to keep results on Drive across disconnects.

**Runtime:** `Runtime → Change runtime type → GPU`.

## 1 · Setup

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/Mr0022/ProjectA.git"
BRANCH = "claude/optimistic-mccarthy-6zszk7"
DIR    = "/content/ProjectA"

if not os.path.isdir(DIR):
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO,DIR], check=True)
else:
    subprocess.run(["git","-C",DIR,"fetch","origin",BRANCH], check=True)
    subprocess.run(["git","-C",DIR,"checkout",BRANCH], check=True)
    subprocess.run(["git","-C",DIR,"reset","--hard",f"origin/{BRANCH}"], check=True)

os.chdir(DIR); sys.path.insert(0, DIR)
print("HEAD:", subprocess.run(["git","log","--oneline","-1"],
                              capture_output=True, text=True).stdout.strip())

# Guard: the aggregated target must be log(mean RV) = logsumexp(ln_RV) - log(h).
src = open("exp/exp_ModernTCN.py").read()
assert "math.log(h)" in src and "torch.logsumexp" in src, (
    f"stale checkout — delete {DIR} and re-run this cell")
print("target check: OK  (log(mean RV))")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2 · Anchor and configuration

In [ ]:
PRED_LEN = 5
TAG      = "custom"      # keeps this study in its own CSV / figure folder

# Your anchor, exactly as given.
ANCHOR = {
    "seq_len":       35,
    "patch_size":    32,
    "patch_stride":  2,
    "ffn_ratio":     2,
    "num_blocks":    1,
    "large_size":    31,
    "small_size":    5,
    "dim":           64,
    "dropout":       0.0,
    "head_dropout":  0.0,
    "revin":         1,
    "learning_rate": 0.0077943332090161695,
    "batch_size":    256,
}

# All 13 hyperparameters in the tuning search space.
from sensitivity.ofat_sensitivity import ORDER, GRIDS
PARAMS = list(ORDER)

ITR    = 5     # seeds per point
EPOCHS = 40    # max epochs per seed (early stopping, patience 8)

USE_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/ofat_h5_custom"

ANCHOR_ARGS = [f"{k}={v}" for k, v in ANCHOR.items()]

n = sum(len(set(GRIDS[p]) | {ANCHOR[p]}) - 1 for p in PARAMS) + 1
print(f"h = {PRED_LEN}, {len(PARAMS)} hyperparameters -> {n} configs x {ITR} seeds "
      f"= {n * ITR} training runs")
print("\nGrid per parameter (anchor value in [brackets]):")
for p in PARAMS:
    vals = sorted(set(GRIDS[p]) | {ANCHOR[p]})
    print("  " + f"{p:<14}" + "  ".join(f"[{v:g}]" if v == ANCHOR[p] else f"{v:g}"
                                        for v in vals))

## 3 · Run the sweep

Progress streams below. Interrupt any time and re-run — completed points are skipped.

In [ ]:
import re, time, shutil, glob

CSV = f"sensitivity/ofat_moderntcn_h{PRED_LEN}_{TAG}.csv"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for f in glob.glob(os.path.join(DRIVE_DIR, "ofat_moderntcn_h*")):
        shutil.copy(f, "sensitivity/")
        print("restored", os.path.basename(f))

def save_to_drive():
    if USE_DRIVE:
        for f in glob.glob(f"sensitivity/ofat_moderntcn_h{PRED_LEN}_{TAG}*"):
            shutil.copy(f, DRIVE_DIR)

def stream(cmd, keep=r"^(\[OK\]|\[WARN\]|OFAT plan|   anchor differs|##########)"):
    pat, buf = re.compile(keep), []
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
    for line in pr.stdout:
        buf.append(line)
        if pat.search(line):
            print(line.rstrip())
    pr.wait()
    if pr.returncode != 0:
        print("".join(buf[-25:]))
        raise RuntimeError(f"exit {pr.returncode}")
    return "".join(buf)

t0 = time.time()
try:
    stream([sys.executable, "sensitivity/ofat_sensitivity.py",
            "--pred_len", str(PRED_LEN), "--tag", TAG,
            "--anchor", *ANCHOR_ARGS, "--params", *PARAMS,
            "--itr", str(ITR), "--train_epochs", str(EPOCHS)])
finally:
    save_to_drive()     # keep partial progress even if interrupted
print(f"\nsweep finished in {(time.time()-t0)/60:.1f} min -> {CSV}")

## 4 · Figures

Response curves are one series per panel (mean line, ±1 std band, per-seed dots,
◆ = your anchor). The tornado shows the signed swing each knob can produce, and
the bar ranks them.

In [ ]:
stream([sys.executable, "sensitivity/ofat_plots.py",
        "--pred_len", str(PRED_LEN), "--tag", TAG], keep=r"^(wrote|Figures)")

In [ ]:
from IPython.display import Image, display

FIGDIR = f"sensitivity/figures/h{PRED_LEN}_{TAG}"
for name in ("ofat_response_mse", "ofat_response_qlike",
             "ofat_tornado_mse", "ofat_sensitivity_bar"):
    p = f"{FIGDIR}/{name}.png"
    if os.path.exists(p):
        display(Image(filename=p))

## 5 · Ranked summary

Which knobs move test MSE most around your anchor, and what the best value on the
grid was. This is also the text fallback for anything hard to read off the figures.

In [ ]:
import numpy as np, pandas as pd

s = pd.read_csv(f"{FIGDIR}/ofat_summary.csv")
rows = []
for metric in ("mse", "qlike"):
    m = s[s.metric == metric]
    a = float(m.loc[m.is_anchor.astype(bool), "mean"].mean())
    for prm, g in m.groupby("param"):
        if len(g) < 2:
            continue
        best = g.loc[g["mean"].idxmin()]
        rows.append(dict(metric=metric.upper(), param=prm,
                         anchor=a, best=best["mean"], best_at=best["value"],
                         worst=g["mean"].max(),
                         swing_pct=(g["mean"].max() - g["mean"].min()) / a * 100,
                         gain_vs_anchor_pct=(best["mean"] / a - 1) * 100))
tbl = pd.DataFrame(rows)

for metric in ("MSE", "QLIKE"):
    t = (tbl[tbl.metric == metric]
         .drop(columns="metric")
         .sort_values("swing_pct", ascending=False)
         .set_index("param"))
    print(f"\n=== {metric} — anchor {t['anchor'].iloc[0]:.4f} ===")
    print("swing_pct = spread across the grid, as % of anchor (how much it matters)")
    print("gain_vs_anchor_pct = best grid value vs anchor (negative = better than anchor)\n")
    display(t.round(4))

tbl.to_csv(f"{FIGDIR}/ofat_ranked.csv", index=False)
print(f"\nsaved {FIGDIR}/ofat_ranked.csv")

In [ ]:
import shutil
shutil.make_archive(f"/content/ofat_h{PRED_LEN}_{TAG}", "zip", FIGDIR)
try:
    from google.colab import files
    files.download(f"/content/ofat_h{PRED_LEN}_{TAG}.zip")
except Exception as e:
    print(f"(figures in {os.getcwd()}/{FIGDIR}; "
          f"auto-download unavailable: {type(e).__name__})")

---

### Notes

- **`patch_stride` is clamped** to `min(patch_stride, patch_size)`, exactly as
  `tune.py` does. Your anchor stride is 2, below every grid patch size, so the
  clamp never bites here — but the `patch_size` sweep still cannot produce a
  stride wider than the patch.
- Only `learning_rate` needs an extra grid point: your `0.0077943…` is not one of
  the searched values, so its curve has 6 points instead of 5. Every other anchor
  value already sits on the grid, which is why this is 41 configs rather than more.
- **OFAT is local.** It measures sensitivity *around your anchor*, not a global
  importance ranking. A knob that looks flat here may still matter elsewhere in
  the space; for the global view use the Optuna `param_importances` artifacts in
  `tuningresults/`.
- The command you based this on was labelled `--model_id EventTCN_h22` but passes
  no `--use_events` and `--pred_len 5`. This notebook runs **plain ModernTCN at
  h = 5**, which is what those flags actually do. Add `--use_events` to the sweep
  command if you meant the event model.